In [10]:
import pandas as pd
import numpy as np
from os import getenv
from joblib import load
from datetime import datetime, timedelta
from utils import get_stocks, get_candles, calculate_indicators_for_prediction_modul

In [11]:
tickers , dv, model = load('stocks_analyser_tickers_dv_with_model.joblib')

In [12]:
# выгружаю токен для т-инвестиций
TOKEN = getenv("t_bank_token")
# P.S: если захотите воспользоваться этим модулем, вам придётся создать свой токен на сайте т-банка
# и записать его в эту переменную в таком формате: TOKEN = "ваш_токен"

In [13]:
# получаем данные по акциям
stocks = await get_stocks(TOKEN)

In [14]:
# создаем диапозон дат за 70 дней
end_date = datetime.now()
start_date = end_date - timedelta(days=70)

# выгружаем свечи в этом дипазоне
candles_df = await get_candles(TOKEN, stocks, tickers, start_date, end_date)
candles_df = candles_df.sort_values(['ticker', 'date']).reset_index(drop=True)

Запрашиваем данные с 2026-05-09 по 2026-07-18
Всего тикеров: 153


In [15]:
# добавляем индикаторы
candles_df.date = pd.to_datetime(candles_df.date)
candles_df = calculate_indicators_for_prediction_modul(candles_df)

In [16]:
# удаляем бесконечности и NaN
candles_df.replace([np.inf, -np.inf], np.nan, inplace=True)
candles_df.dropna(inplace=True)
candles_df = candles_df[candles_df.ticker.isin(tickers)]
candles_df

,ticker,sector,date,open,high,low,close,volume,season,atr,...,volatility_20,volatility_60,vol_ratio_20_60,momentum,range_norm,range_ratio,open_close_ratio,volume_spike,ad_line,return
69,ABRD,consumer,2026-05-13,144.80,146.00,144.60,144.80,757,spring,6.689908,...,0.776468,0.449045,1.729154,4.452645,0.142857,1.009682,1.000000,0,-2.666728e+05,-0.001379
70,ABRD,consumer,2026-05-14,145.20,145.80,143.20,143.40,1631,spring,6.495150,...,0.776102,0.449060,1.728280,4.412308,0.076923,1.018156,1.012552,0,-2.680528e+05,-0.009669
71,ABRD,consumer,2026-05-15,143.60,143.80,141.60,142.00,2369,spring,6.290619,...,0.776005,0.449104,1.727894,0.979310,0.181818,1.015537,1.011268,0,-2.695604e+05,-0.009763
72,ABRD,consumer,2026-05-16,143.40,143.40,142.20,142.40,69,spring,6.057733,...,0.775923,0.449104,1.727712,0.982069,0.166667,1.008439,1.007022,0,-2.696064e+05,0.002817
73,ABRD,consumer,2026-05-17,142.80,142.80,142.20,142.40,86,spring,5.797841,...,0.776043,0.449043,1.728217,0.983425,0.333333,1.004219,1.002809,0,-2.696351e+05,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10106,ZAYM,financial,2026-07-14,130.55,132.25,130.55,132.05,3052,summer,15.230743,...,0.020604,0.012702,1.622156,1.024438,0.882353,1.013022,0.988641,0,-2.398511e+09,0.011877
10107,ZAYM,financial,2026-07-15,132.05,132.95,130.50,130.55,8488,summer,14.622137,...,0.018439,0.012715,1.450226,1.016745,0.020408,1.018774,1.011490,0,-2.398519e+09,-0.011359
10108,ZAYM,financial,2026-07-16,130.50,130.60,124.50,126.40,17273,summer,14.216320,...,0.010336,0.013266,0.779106,0.977572,0.311475,1.048996,1.032437,1,-2.398526e+09,-0.031789
10109,ZAYM,financial,2026-07-17,126.40,127.80,121.05,122.30,10782,summer,13.860781,...,0.011916,0.013797,0.863687,0.937165,0.185185,1.055762,1.033524,0,-2.398532e+09,-0.032437


In [17]:
# проводим ручную фильтрацию по техническим индиакторам для тестовой выборки
candles_df = candles_df[
    (candles_df.sma10_vs_sma40 > 1.02) &
    (candles_df['hist'] > candles_df.signal) &
    (candles_df.stoch_k > candles_df.stoch_d) &
    (candles_df['volume_ratio'] > 1.2) &
    (candles_df['volume_spike'] == 1) &
    (candles_df['ema_12'] > candles_df['ema_26']) &
    (candles_df['price_vs_sma_10'] > 1.01)
    ]

In [18]:
# убираем колонки, которые нельзя использовать в прогнозировании
features_col = [col for col in candles_df.columns if col not in ['index', 'open', 'high', 'low', 'close', 'volume','date']]

In [19]:
# формируем матрицу признаков
candles_dict = candles_df[features_col].to_dict(orient='records')
X = dv.transform(candles_dict)

In [20]:
# формирование предсказаний
proba = model.predict_proba(X)[:, 1] > 0.6

In [21]:
# Если есть свежие прогнозы, выводим их на экран
predictions = candles_df[proba & (candles_df.date == candles_df.date.max())].T

if (candles_df.date.max() == datetime.now()) & (len(predictions.T) > 0):
    display(predictions)
else:
    print("Нет прогнозов на эту дату")

Нет прогнозов на эту дату
